# AlphaFold2 Structure Prediction Example

This notebook demonstrates how to predict protein 3D structures using **AlphaFold2** via the ColabDesign JAX wrapper.

## What is AlphaFold2?

AlphaFold2 is the original highly accurate protein structure prediction model from DeepMind. It uses an attention-based architecture with optional multiple sequence alignments (MSAs) and recycling iterations to predict protein structures.

### Key Features:
- **High accuracy** -- State-of-the-art structure prediction
- **MSA support** -- Optional evolutionary information via ColabFold search
- **Multi-chain** -- Supports protein complexes via AlphaFold-Multimer
- **Confidence scores** -- pLDDT, pTM, iPTM, and PAE metrics

## Imports

## Shared Data Types

### `StructurePredictionComplex`
A biological complex containing one or more molecular chains to predict together. AlphaFold2 only supports protein chains.

| Field | Type | Description |
|-------|------|-------------|
| `chains` | `List[Chain]` | Protein chains in the complex. Accepts strings or `Chain` objects |

### `Structure`
A predicted 3D structure with coordinates, confidence metrics, and export methods.

In [1]:
from pathlib import Path

from proto_tools import AlphaFold2Config, AlphaFold2Input, run_alphafold2

## Single Protein Structure Prediction

Predict a single protein structure without MSA (single-sequence mode).

### API Reference

**`AlphaFold2Input`**

| Field | Type | Description |
|-------|------|-------------|
| `complexes` | `List[StructurePredictionComplex]` | Complexes to predict. Each complex contains one or more protein chains |

**`AlphaFold2Config`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `use_msa` | `bool` | `True` | Whether to generate and use MSAs via ColabFold search |
| `num_ensemble_models` | `int` | `1` | Number of model parameter sets to run and average (1-5) |
| `num_recycles` | `int` | `3` | Number of recycling iterations (0-48) |
| `model_num` | `int` | `1` | Which AlphaFold2 model parameter set to use (1-5). Mutually exclusive with `num_ensemble_models > 1` |
| `seed` | `Optional[int]` | `None` | Random seed for reproducibility |
| `device` | `str` | `"cuda"` | Device to run on (`"cuda"`, `"cpu"`) |

**`AlphaFold2Output`**

| Field | Type | Description |
|-------|------|-------------|
| `structures` | `List[Structure]` | Predicted structures, one per input complex. Each structure contains 3D coordinates and confidence metrics (`avg_plddt`, `ptm`, `iptm`, `avg_pae`). pLDDT is normalized to 0.0-1.0 scale |

In [2]:
# Small test protein
protein_sequence = "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH"

# Create input
inputs = AlphaFold2Input(complexes=[protein_sequence])

# Configure AlphaFold2 (no MSA for fast prediction)
config = AlphaFold2Config(
    use_msa=False,
    num_recycles=3,
    device="cuda",
)

# Run prediction
result = run_alphafold2(inputs, config)
structure = result.structures[0]

# Print metrics
print(f"  Length: {structure.num_residues} residues")
print(f"  Average pLDDT: {structure.avg_plddt:.3f}")
print(f"  pTM score: {structure.ptm:.3f}")

Folding structures (AlphaFold2): 100%|██████████| 1/1 [01:58<00:00, 118.01s/structure]

  Length: 51 residues
  Average pLDDT: 0.610
  pTM score: 0.338


## Multi-Chain Complex Prediction

Predict a protein-protein complex using AlphaFold-Multimer:

In [3]:
# Two-chain protein complex
chain_a = "MARFLGLGARYTWM"
chain_b = "YTWHKLARFGMVLS"

# Create multi-chain input
inputs = AlphaFold2Input(complexes=[[chain_a, chain_b]])

# Configure (multimer mode is automatic when >1 chain)
config = AlphaFold2Config(
    use_msa=False,
    num_recycles=3,
    device="cuda",
)

# Run prediction
result = run_alphafold2(inputs, config)
complex_structure = result.structures[0]

# Print metrics (iPTM is available for multimer)
print(f"  Chains: {complex_structure.num_chains}")
print(f"  Average pLDDT: {complex_structure.avg_plddt:.3f}")
print(f"  pTM score: {complex_structure.ptm:.3f}")
print(f"  iPTM score: {complex_structure.metrics.get('iptm', 'N/A')}")

Folding structures (AlphaFold2): 100%|██████████| 1/1 [00:43<00:00, 43.72s/structure]

  Chains: 2
  Average pLDDT: 0.438
  pTM score: 0.112
  iPTM score: 0.0181519016623497


## MSA-Assisted Prediction

For higher accuracy, enable MSA generation via ColabFold search:

In [4]:
# Predict with MSA (uses ColabFold search for evolutionary information)
protein_sequence = "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH"

inputs = AlphaFold2Input(complexes=[protein_sequence])
config = AlphaFold2Config(
    use_msa=True,  # Enable MSA generation
    num_recycles=3,
    device="cuda",
    verbose=True,
)

result = run_alphafold2(inputs, config)
structure = result.structures[0]

print(f"  Average pLDDT: {structure.avg_plddt:.3f}")
print(f"  pTM score: {structure.ptm:.3f}")

COMPLETE: 100%|██████████| 150/150 [elapsed: 00:01 remaining: 00:00]
DEBUG | Predicting structure: 1 chain(s), 51 residues, multimer=False, device=cuda
DEBUG | MSA injected: 3251 sequences, length=51
/home/bviggiano/codebases/proto-language/proto-tools/tool_envs/alphafold2_env/lib/python3.12/site-packages/colabdesign/af/alphafold/model/msa.py:91: UserWarning: Explicitly requested dtype int64 requested in broadcasted_iota is not available, and will be truncated to dtype int32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  iota = jax.lax.broadcasted_iota(jnp.int64, logits.shape, axis)
DEBUG | Prediction complete: avg_plddt=0.9259, ptm=0.5888


predict models [0] recycles 3 loss 2.93 con 2.93 plddt 0.93 ptm 0.59 rmsd 0.00


Folding structures (AlphaFold2): 100%|██████████| 1/1 [00:47<00:00, 47.10s/structure]

  Average pLDDT: 0.926
  pTM score: 0.589


## Export Results

Save predicted structures for further analysis in PyMOL, ChimeraX, or VMD.

### Supported formats:
- **PDB** -- Standard protein data bank format
- **mmCIF** -- Modern crystallographic information file

The B-factor column contains normalized pLDDT scores (0.0-1.0).

In [5]:
# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to PDB files
result.export(name="alphafold2_structure", export_path=output_dir, file_format="pdb")